In [ ]:
import imaplib
import email
import re
import pandas as pd
import unicodedata
from bs4 import BeautifulSoup
from email.header import decode_header

In [33]:
# ==========================================
# CONFIGURACION
# ==========================================

# Normalizar texto de asunto para comparaciones
def normalizar_texto(texto):
    texto = texto.lower()

    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    return texto.strip()

In [34]:
# ==========================================
# CONFIGURACION
# ==========================================

EMAIL = "diego.vaca.enriquez@gmail.com"
PASSWORD = "xbfk lybw jznn yzdc"

FECHA_DESDE = "01-Aug-2026"
ASUNTO_OBJETIVO = ("notificacion de consumos")

In [35]:
# ==========================================
# CONEXION A GMAIL
# ==========================================

mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")

('OK', [b'38985'])

In [36]:
# Buscar correos recientes
status, messages = mail.search(
    None,
    f'(SINCE "{FECHA_DESDE}")'
)

ids = messages[0].split()

print(f"Correos encontrados desde {FECHA_DESDE}: {len(ids)}")

Correos encontrados desde 01-Aug-2026: 150


In [37]:
# ==========================================
# FILTRAR POR ASUNTO
# ==========================================

ids_filtrados = []

for num in ids:

    try:
        status, data = mail.fetch(
            num,
            "(BODY.PEEK[HEADER.FIELDS (SUBJECT)])"
        )

        header_bytes = data[0][1]

        msg_header = email.message_from_bytes(header_bytes)

        asunto = msg_header.get("Subject", "")

        asunto_decode = ""

        for contenido, encoding in decode_header(asunto):

            if isinstance(contenido, bytes):
                asunto_decode += contenido.decode(
                    encoding or "utf-8",
                    errors="ignore"
                )
            else:
                asunto_decode += contenido


        asunto_normalizado = normalizar_texto(asunto_decode)

        if ASUNTO_OBJETIVO in asunto_normalizado:
            ids_filtrados.append(num)
        

    except Exception as e:
        print(f"Error leyendo asunto: {e}")

print(f"Correos con asunto '{ASUNTO_OBJETIVO}': {len(ids_filtrados)}")


Correos con asunto 'notificacion de consumos': 1


In [31]:
# ==========================================
# EXTRACCION DE DATOS
# ==========================================

compras = []

for num in ids_filtrados:

    try:

        status, data = mail.fetch(num, "(RFC822)")

        msg = email.message_from_bytes(data[0][1])

        html = None

        if msg.is_multipart():

            for part in msg.walk():

                content_type = part.get_content_type()

                if content_type == "text/html":

                    html = part.get_payload(
                        decode=True
                    ).decode(
                        errors="ignore"
                    )

                    break

        else:

            html = msg.get_payload(
                decode=True
            ).decode(
                errors="ignore"
            )

        if not html:
            continue

        texto = BeautifulSoup(
            html,
            "html.parser"
        ).get_text(" ", strip=True)

        registro = {}

        # ----------------------------------
        # Nombre de la tarjeta
        # ----------------------------------

        m = re.search(
            r"Realizaste una compra con tu\s+(.+?)\s*\.",
            texto,
            re.IGNORECASE | re.DOTALL
        )

        registro["tarjeta"] = (
            m.group(1).strip()
            if m else None
        )

        # ----------------------------------
        # Terminacion tarjeta
        # ----------------------------------

        m = re.search(
            r"Tarjeta terminada en\s+(\d+)",
            texto,
            re.IGNORECASE
        )

        registro["terminacion"] = (
            m.group(1)
            if m else None
        )

        # ----------------------------------
        # Fecha y hora
        # ----------------------------------

        m = re.search(
            r"Fecha\s+(\d{4}-\d{2}-\d{2})\s+(\d{2}:\d{2})",
            texto,
            re.IGNORECASE | re.DOTALL
        )

        registro["fecha"] = (
            m.group(1)
            if m else None
        )

        registro["hora"] = (
            m.group(2)
            if m else None
        )

        # ----------------------------------
        # Establecimiento
        # ----------------------------------

        m = re.search(
            r"Establecimiento\s+(.+?)\s+Valor",
            texto,
            re.IGNORECASE | re.DOTALL
        )

        registro["establecimiento"] = (
            " ".join(m.group(1).split())
            if m else None
        )

        # ----------------------------------
        # Valor
        # ----------------------------------

        m = re.search(
            r"Valor\s+([\d.,]+)",
            texto,
            re.IGNORECASE
        )

        registro["valor"] = (
            m.group(1)
            if m else None
        )

        compras.append(registro)

    except Exception as e:
        print(f"Error procesando correo {num}: {e}")

# ==========================================
# RESULTADOS
# ==========================================

df = pd.DataFrame(compras)

print()
print(df.head())
print()
print(f"Compras extraídas: {len(df)}")

df.to_excel(
    "output/consumos_tarjeta.xlsx",
    index=False
)

print("Archivo generado: consumos_tarjeta.xlsx")

Error procesando correo b'38887': name 'BeautifulSoup' is not defined

Empty DataFrame
Columns: []
Index: []

Compras extraídas: 0


ModuleNotFoundError: No module named 'openpyxl'